In [1]:
import os
%pwd

'/home/tuhin/bangla-political-memes-classification/research'

In [2]:
import os
if os.path.basename(os.getcwd()) == 'research':
    os.chdir("../")

In [3]:
%pwd

'/home/tuhin/bangla-political-memes-classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TextPreprocessingConfig:
    root_dir: Path
    input_train_csv: Path
    input_test_csv: Path
    output_train_csv: Path
    output_test_csv: Path

In [5]:
from memeClassifier.constants import *
from memeClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_text_preprocessing_config(self) -> TextPreprocessingConfig:
        config = self.config.text_preprocessing

        create_directories([config.root_dir])

        text_preprocessing_config = TextPreprocessingConfig(
            root_dir=Path(config.root_dir),
            input_train_csv=Path(config.input_train_csv),
            input_test_csv=Path(config.input_test_csv),
            output_train_csv=Path(config.output_train_csv),
            output_test_csv=Path(config.output_test_csv)
        )

        return text_preprocessing_config

In [7]:
import re
import pandas as pd
from bnunicodenormalizer import Normalizer
from spellchecker import SpellChecker
from memeClassifier import logger

class TextPreprocessing:
    def __init__(self, config: TextPreprocessingConfig):
        self.config = config
        self.bnorm = Normalizer()
        self.spell = SpellChecker()

    def preprocess_text(self, text):
        if not isinstance(text, str) or text == '':
            return ""
        
        text = text.lower()
        text = re.sub(r'[\|\{\}\[\]\(\);]+', ' ', text)
        text = re.sub(r'\b\w*\d+_\w*\b', ' ', text)
        text = re.sub(r'\b\d+_\d+\b', ' ', text)
        text = re.sub(r'\b\w*\d+\w*_\b', ' ', text)
        text = re.sub(r'\b_\d+\w*\b', ' ', text)
        text = re.sub(r'\b\d+[a-z]*\b', ' ', text)
        text = re.sub(r'\b[a-z]*\d+\b', ' ', text)
        text = re.sub(r'\b[\u09E6-\u09EF]+\b', ' ', text)
        text = re.sub(r'\b[\u09E6-\u09EF]+[\u0980-\u09FF]+\b', ' ', text)
        text = re.sub(r'\b[\u0980-\u09FF]+[\u09E6-\u09EF]+\b', ' ', text)
        text = re.sub(r'\b[\u0980-\u09FF]*[\u09E6-\u09EF]+[\u0980-\u09FF]+[\u09E6-\u09EF]*\b', ' ', text)
        text = re.sub(r'[^\w\s\u0980-\u09FF.,!?]', ' ', text)
        text = re.sub(r'\b[a-z]\b', ' ', text, flags=re.IGNORECASE)
        text = re.sub(r'\b[\u0980-\u09FF]\b', ' ', text)

        words = text.split()
        cleaned_words = []
        
        for word in words:
            if len(word) < 2 or word.strip('.,!?') == '':
                continue
            if '_' in word or re.search(r'\d', word):
                continue
            if re.search(r'[\u09E6-\u09EF]', word):
                continue
            has_bengali = bool(re.search(r'[\u0980-\u09FF]', word))
            if has_bengali:
                try:
                    normalized = self.bnorm(word)['normalized']
                    if normalized and len(normalized) > 1:
                        cleaned_words.append(normalized)
                except:
                    if len(word) > 1:
                        cleaned_words.append(word)
            else:
                word_clean = word.strip('.,!?')
                if len(word_clean) > 2:
                    try:
                        corrected = self.spell.correction(word_clean)
                        if corrected and corrected != word_clean:
                            if corrected in self.spell:
                                cleaned_words.append(corrected)
                            else:
                                cleaned_words.append(word_clean)
                        else:
                            cleaned_words.append(word_clean)
                    except:
                        cleaned_words.append(word_clean)
        
        text = ' '.join(cleaned_words)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def initiate_text_preprocessing(self):
        logger.info("Starting text preprocessing for train and test datasets")
        
        train_df = pd.read_csv(self.config.input_train_csv)
        test_df = pd.read_csv(self.config.input_test_csv)
        
        logger.info("Preprocessing train dataset...")
        train_df['Processed_Text'] = train_df['Extracted_Text'].apply(self.preprocess_text)
        
        logger.info("Preprocessing test dataset...")
        test_df['Processed_Text'] = test_df['Extracted_Text'].apply(self.preprocess_text)
        
        train_df.to_csv(self.config.output_train_csv, index=False)
        test_df.to_csv(self.config.output_test_csv, index=False)
        
        logger.info(f"Saved processed train dataset to {self.config.output_train_csv}")
        logger.info(f"Saved processed test dataset to {self.config.output_test_csv}")

In [8]:
try:
    config = ConfigurationManager()
    text_preprocessing_config = config.get_text_preprocessing_config()
    text_preprocessing = TextPreprocessing(config=text_preprocessing_config)
    text_preprocessing.initiate_text_preprocessing()
except Exception as e:
    raise e

[2026-06-27 02:22:43,579: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-27 02:22:43,581: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 02:22:43,582: INFO: common: Directory created at: artifacts]
[2026-06-27 02:22:43,582: INFO: common: Directory created at: artifacts/text_preprocessing]
[2026-06-27 02:22:43,703: INFO: 239461760: Starting text preprocessing for train and test datasets]
[2026-06-27 02:22:43,708: INFO: 239461760: Preprocessing train dataset...]
[2026-06-27 02:24:30,371: INFO: 239461760: Preprocessing test dataset...]
[2026-06-27 02:25:36,951: INFO: 239461760: Saved processed train dataset to artifacts/text_preprocessing/train_dataset_cleaned.csv]
[2026-06-27 02:25:36,952: INFO: 239461760: Saved processed test dataset to artifacts/text_preprocessing/test_dataset_cleaned.csv]
